In [ ]:
"""
Comprehensive ANOVA Analysis for India EXIM Critical Minerals
Statistical testing for trade pattern differences across:
- Time periods (years, quarters)
- Mineral categories
- Trading partners
- Import vs Export dynamics
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
from statsmodels.graphics.factorplots import interaction_plot
import warnings
warnings.filterwarnings('ignore')

============================================================================
SECTION 1: ONE-WAY ANOVA
============================================================================

In [ ]:
class OneWayANOVA:
    """Perform one-way ANOVA analysis"""
    
    def __init__(self, data, dependent_var, independent_var):
        self.data = data
        self.dependent_var = dependent_var
        self.independent_var = independent_var
        self.results = None
        
    def check_assumptions(self):
        """Check ANOVA assumptions"""
        print("="*70)
        print("CHECKING ANOVA ASSUMPTIONS")
        print("="*70)
        
        groups = self.data.groupby(self.independent_var)[self.dependent_var]
        
        # 1. Normality test (Shapiro-Wilk)
        print("\n1. NORMALITY TEST (Shapiro-Wilk)")
        print("-" * 50)
        
        for name, group in groups:
            if len(group) >= 3:
                stat, p_value = stats.shapiro(group.dropna())
                result = "Normal" if p_value > 0.05 else "Not Normal"
                print(f"{name}: W={stat:.4f}, p={p_value:.4f} [{result}]")
        
        # 2. Homogeneity of variance (Levene's test)
        print("\n2. HOMOGENEITY OF VARIANCE (Levene's Test)")
        print("-" * 50)
        
        group_data = [group.dropna().values for name, group in groups]
        stat, p_value = stats.levene(*group_data)
        result = "Equal variances" if p_value > 0.05 else "Unequal variances"
        print(f"Levene's statistic: {stat:.4f}")
        print(f"p-value: {p_value:.4f}")
        print(f"Result: {result}")
        
        # 3. Sample size check
        print("\n3. SAMPLE SIZE")
        print("-" * 50)
        print(groups.size())
        
        return {
            'levene_stat': stat,
            'levene_pvalue': p_value,
            'equal_variances': p_value > 0.05
        }
    
    def perform_anova(self):
        """Perform one-way ANOVA"""
        print("\n" + "="*70)
        print("ONE-WAY ANOVA RESULTS")
        print("="*70)
        
        # Prepare data
        groups = self.data.groupby(self.independent_var)[self.dependent_var]
        group_data = [group.dropna().values for name, group in groups]
        
        # Perform ANOVA
        f_stat, p_value = stats.f_oneway(*group_data)
        
        print(f"\nF-statistic: {f_stat:.4f}")
        print(f"p-value: {p_value:.6f}")
        
        if p_value < 0.001:
            significance = "Highly Significant (***)"
        elif p_value < 0.01:
            significance = "Very Significant (**)"
        elif p_value < 0.05:
            significance = "Significant (*)"
        else:
            significance = "Not Significant (ns)"
        
        print(f"\nResult: {significance}")
        
        if p_value < 0.05:
            print("\n✓ There are significant differences between groups")
            print("  → Post-hoc tests recommended")
        else:
            print("\n✗ No significant differences between groups")
        
        self.results = {
            'f_statistic': f_stat,
            'p_value': p_value,
            'significance': significance,
            'reject_null': p_value < 0.05
        }
        
        return self.results
    
    def effect_size(self):
        """Calculate effect size (eta-squared)"""
        groups = self.data.groupby(self.independent_var)[self.dependent_var]
        
        # Calculate sum of squares
        grand_mean = self.data[self.dependent_var].mean()
        
        # Between-group sum of squares
        ss_between = sum(len(group) * (group.mean() - grand_mean)**2 
                        for name, group in groups)
        
        # Total sum of squares
        ss_total = sum((self.data[self.dependent_var] - grand_mean)**2)
        
        # Eta-squared
        eta_squared = ss_between / ss_total
        
        # Interpret effect size
        if eta_squared < 0.01:
            interpretation = "Small"
        elif eta_squared < 0.06:
            interpretation = "Medium"
        else:
            interpretation = "Large"
        
        print("\n" + "="*70)
        print("EFFECT SIZE")
        print("="*70)
        print(f"η² (Eta-squared): {eta_squared:.4f}")
        print(f"Interpretation: {interpretation} effect")
        print(f"Variance explained: {eta_squared*100:.2f}%")
        
        return eta_squared
    
    def descriptive_statistics(self):
        """Calculate descriptive statistics by group"""
        print("\n" + "="*70)
        print("DESCRIPTIVE STATISTICS BY GROUP")
        print("="*70)
        
        desc_stats = self.data.groupby(self.independent_var)[self.dependent_var].describe()
        print("\n", desc_stats)
        
        # Additional statistics
        print("\n" + "="*70)
        print("ADDITIONAL STATISTICS")
        print("="*70)
        
        groups = self.data.groupby(self.independent_var)[self.dependent_var]
        
        additional_stats = pd.DataFrame({
            'median': groups.median(),
            'variance': groups.var(),
            'skewness': groups.skew(),
            'kurtosis': groups.apply(lambda x: x.kurtosis())
        })
        
        print("\n", additional_stats)
        
        return desc_stats, additional_stats
    
    def visualize_results(self):
        """Create visualization of ANOVA results"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 1. Box plot
        self.data.boxplot(column=self.dependent_var, 
                         by=self.independent_var, 
                         ax=axes[0, 0])
        axes[0, 0].set_title('Box Plot by Group')
        axes[0, 0].set_xlabel(self.independent_var)
        axes[0, 0].set_ylabel(self.dependent_var)
        plt.sca(axes[0, 0])
        plt.xticks(rotation=45)
        
        # 2. Violin plot
        sns.violinplot(data=self.data, 
                      x=self.independent_var, 
                      y=self.dependent_var, 
                      ax=axes[0, 1])
        axes[0, 1].set_title('Violin Plot by Group')
        axes[0, 1].tick_params(axis='x', rotation=45)
        
        # 3. Mean with confidence intervals
        means = self.data.groupby(self.independent_var)[self.dependent_var].mean()
        stds = self.data.groupby(self.independent_var)[self.dependent_var].std()
        sems = self.data.groupby(self.independent_var)[self.dependent_var].sem()
        
        x_pos = np.arange(len(means))
        axes[1, 0].bar(x_pos, means, yerr=1.96*sems, capsize=5, alpha=0.7)
        axes[1, 0].set_xticks(x_pos)
        axes[1, 0].set_xticklabels(means.index, rotation=45)
        axes[1, 0].set_title('Mean with 95% CI')
        axes[1, 0].set_ylabel(self.dependent_var)
        
        # 4. Distribution plot
        for name, group in self.data.groupby(self.independent_var):
            axes[1, 1].hist(group[self.dependent_var].dropna(), 
                          alpha=0.5, label=name, bins=20)
        axes[1, 1].set_title('Distribution by Group')
        axes[1, 1].set_xlabel(self.dependent_var)
        axes[1, 1].legend()
        
        plt.tight_layout()
        return fig

============================================================================
SECTION 2: POST-HOC TESTS
============================================================================

In [ ]:
class PostHocTests:
    """Perform post-hoc tests after ANOVA"""
    
    @staticmethod
    def tukey_hsd(data, dependent_var, independent_var):
        """Tukey's Honestly Significant Difference test"""
        print("\n" + "="*70)
        print("TUKEY'S HSD POST-HOC TEST")
        print("="*70)
        
        # Perform Tukey's HSD
        tukey = pairwise_tukeyhsd(
            endog=data[dependent_var],
            groups=data[independent_var],
            alpha=0.05
        )
        
        print("\n", tukey)
        
        # Create visualization
        fig, ax = plt.subplots(figsize=(10, 8))
        tukey.plot_simultaneous(ax=ax)
        plt.title("Tukey's HSD: 95% Confidence Intervals")
        plt.tight_layout()
        
        return tukey, fig
    
    @staticmethod
    def bonferroni_correction(data, dependent_var, independent_var):
        """Bonferroni correction for multiple comparisons"""
        print("\n" + "="*70)
        print("BONFERRONI CORRECTION")
        print("="*70)
        
        groups = data[independent_var].unique()
        n_comparisons = len(groups) * (len(groups) - 1) / 2
        alpha_corrected = 0.05 / n_comparisons
        
        print(f"Number of comparisons: {int(n_comparisons)}")
        print(f"Corrected α: {alpha_corrected:.6f}")
        
        # Pairwise comparisons
        results = []
        for i, group1 in enumerate(groups):
            for group2 in groups[i+1:]:
                data1 = data[data[independent_var] == group1][dependent_var].dropna()
                data2 = data[data[independent_var] == group2][dependent_var].dropna()
                
                t_stat, p_value = stats.ttest_ind(data1, data2)
                
                results.append({
                    'Group1': group1,
                    'Group2': group2,
                    't-statistic': t_stat,
                    'p-value': p_value,
                    'Significant (Bonferroni)': p_value < alpha_corrected
                })
        
        results_df = pd.DataFrame(results)
        print("\n", results_df.to_string(index=False))
        
        return results_df

============================================================================
SECTION 3: TWO-WAY ANOVA
============================================================================

In [ ]:
class TwoWayANOVA:
    """Perform two-way ANOVA analysis"""
    
    def __init__(self, data, dependent_var, factor1, factor2):
        self.data = data
        self.dependent_var = dependent_var
        self.factor1 = factor1
        self.factor2 = factor2
        
    def perform_anova(self):
        """Perform two-way ANOVA with interaction"""
        print("\n" + "="*70)
        print("TWO-WAY ANOVA RESULTS")
        print("="*70)
        
        # Create formula
        formula = f'{self.dependent_var} ~ C({self.factor1}) + C({self.factor2}) + C({self.factor1}):C({self.factor2})'
        
        # Fit model
        model = ols(formula, data=self.data).fit()
        
        # ANOVA table
        anova_table = anova_lm(model, typ=2)
        
        print("\n", anova_table)
        
        # Interpret results
        print("\n" + "="*70)
        print("INTERPRETATION")
        print("="*70)
        
        for effect in [f'C({self.factor1})', f'C({self.factor2})', 
                      f'C({self.factor1}):C({self.factor2})']:
            if effect in anova_table.index:
                p_val = anova_table.loc[effect, 'PR(>F)']
                f_val = anova_table.loc[effect, 'F']
                
                if p_val < 0.001:
                    sig = "Highly Significant (***)"
                elif p_val < 0.01:
                    sig = "Very Significant (**)"
                elif p_val < 0.05:
                    sig = "Significant (*)"
                else:
                    sig = "Not Significant (ns)"
                
                print(f"\n{effect}:")
                print(f"  F = {f_val:.4f}, p = {p_val:.6f}")
                print(f"  {sig}")
        
        return anova_table, model
    
    def interaction_plot(self):
        """Plot interaction effects"""
        fig, ax = plt.subplots(figsize=(10, 6))
        
        interaction_plot(
            x=self.data[self.factor1],
            trace=self.data[self.factor2],
            response=self.data[self.dependent_var],
            ax=ax
        )
        
        plt.title(f'Interaction Plot: {self.factor1} × {self.factor2}')
        plt.xlabel(self.factor1)
        plt.ylabel(f'Mean {self.dependent_var}')
        plt.tight_layout()
        
        return fig

============================================================================
SECTION 4: REPEATED MEASURES ANOVA
============================================================================

In [ ]:
class RepeatedMeasuresANOVA:
    """Repeated measures ANOVA for time series data"""
    
    def __init__(self, data, dependent_var, within_factor, subject_id):
        self.data = data
        self.dependent_var = dependent_var
        self.within_factor = within_factor
        self.subject_id = subject_id
    
    def check_sphericity(self):
        """Check sphericity assumption (Mauchly's test)"""
        print("\n" + "="*70)
        print("SPHERICITY ASSUMPTION CHECK")
        print("="*70)
        print("Note: Sphericity test requires specialized libraries")
        print("For manual verification, check correlation matrices")
        
        # Calculate correlations between levels
        pivot_data = self.data.pivot(
            index=self.subject_id,
            columns=self.within_factor,
            values=self.dependent_var
        )
        
        corr_matrix = pivot_data.corr()
        print("\nCorrelation Matrix:")
        print(corr_matrix)
        
        return corr_matrix
    
    def perform_rm_anova(self):
        """Perform repeated measures ANOVA"""
        print("\n" + "="*70)
        print("REPEATED MEASURES ANOVA")
        print("="*70)
        
        # Prepare data
        pivot_data = self.data.pivot(
            index=self.subject_id,
            columns=self.within_factor,
            values=self.dependent_var
        )
        
        # Perform Friedman test (non-parametric alternative)
        statistic, p_value = stats.friedmanchisquare(*[pivot_data[col].dropna() 
                                                       for col in pivot_data.columns])
        
        print(f"\nFriedman Chi-square: {statistic:.4f}")
        print(f"p-value: {p_value:.6f}")
        
        if p_value < 0.05:
            print("\n✓ Significant differences across time periods")
        else:
            print("\n✗ No significant differences across time periods")
        
        return statistic, p_value

============================================================================
SECTION 5: COMPREHENSIVE EXIM ANALYSIS
============================================================================

In [ ]:
class EXIMANOVAAnalysis:
    """Comprehensive ANOVA analysis for EXIM data"""
    
    def __init__(self, data):
        self.data = data
        self.results = {}
    
    def analyze_minerals_across_years(self):
        """Compare trade values across different years"""
        print("\n" + "="*70)
        print("ANALYSIS 1: MINERAL TRADE ACROSS YEARS")
        print("="*70)
        
        # One-way ANOVA for each mineral
        for mineral in self.data['mineral'].unique():
            print(f"\n{'='*70}")
            print(f"Analyzing: {mineral}")
            print('='*70)
            
            mineral_data = self.data[self.data['mineral'] == mineral].copy()
            
            anova = OneWayANOVA(mineral_data, 'import_value', 'year')
            anova.check_assumptions()
            results = anova.perform_anova()
            anova.effect_size()
            anova.descriptive_statistics()
            
            # Store results
            self.results[f'{mineral}_by_year'] = results
            
            # Visualize
            fig = anova.visualize_results()
            plt.savefig(f'{mineral}_by_year_anova.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            # Post-hoc if significant
            if results['reject_null']:
                PostHocTests.tukey_hsd(mineral_data, 'import_value', 'year')
    
    def analyze_minerals_comparison(self):
        """Compare different minerals"""
        print("\n" + "="*70)
        print("ANALYSIS 2: COMPARISON ACROSS MINERALS")
        print("="*70)
        
        anova = OneWayANOVA(self.data, 'import_value', 'mineral')
        anova.check_assumptions()
        results = anova.perform_anova()
        anova.effect_size()
        anova.descriptive_statistics()
        
        fig = anova.visualize_results()
        plt.savefig('minerals_comparison_anova.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        if results['reject_null']:
            tukey, fig = PostHocTests.tukey_hsd(self.data, 'import_value', 'mineral')
            plt.savefig('minerals_tukey_hsd.png', dpi=300, bbox_inches='tight')
            plt.close()
        
        self.results['minerals_comparison'] = results
    
    def analyze_seasonal_patterns(self):
        """Analyze seasonal patterns in trade"""
        print("\n" + "="*70)
        print("ANALYSIS 3: SEASONAL PATTERNS")
        print("="*70)
        
        if 'quarter' in self.data.columns:
            anova = OneWayANOVA(self.data, 'import_value', 'quarter')
            anova.check_assumptions()
            results = anova.perform_anova()
            anova.effect_size()
            
            fig = anova.visualize_results()
            plt.savefig('seasonal_patterns_anova.png', dpi=300, bbox_inches='tight')
            plt.close()
            
            self.results['seasonal_patterns'] = results
    
    def analyze_two_way_interaction(self):
        """Analyze mineral × year interaction"""
        print("\n" + "="*70)
        print("ANALYSIS 4: TWO-WAY INTERACTION (Mineral × Year)")
        print("="*70)
        
        two_way = TwoWayANOVA(self.data, 'import_value', 'mineral', 'year')
        anova_table, model = two_way.perform_anova()
        
        fig = two_way.interaction_plot()
        plt.savefig('mineral_year_interaction.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        self.results['two_way_interaction'] = anova_table
    
    def generate_report(self):
        """Generate comprehensive ANOVA report"""
        print("\n" + "="*70)
        print("COMPREHENSIVE ANOVA REPORT SUMMARY")
        print("="*70)
        
        report = []
        
        for analysis_name, result in self.results.items():
            if isinstance(result, dict) and 'f_statistic' in result:
                report.append({
                    'Analysis': analysis_name,
                    'F-statistic': f"{result['f_statistic']:.4f}",
                    'p-value': f"{result['p_value']:.6f}",
                    'Significance': result['significance'],
                    'Decision': 'Reject H₀' if result['reject_null'] else 'Fail to reject H₀'
                })
        
        report_df = pd.DataFrame(report)
        print("\n", report_df.to_string(index=False))
        
        # Save report
        report_df.to_csv('anova_summary_report.csv', index=False)
        print("\n✓ Report saved as 'anova_summary_report.csv'")
        
        return report_df

============================================================================
SECTION 6: MAIN EXECUTION
============================================================================

In [ ]:
def main():
    """Main execution pipeline for ANOVA analysis"""
    
    print("="*70)
    print("COMPREHENSIVE ANOVA ANALYSIS FOR INDIA EXIM DATA")
    print("="*70)
    
    # Load data (sample data for demonstration)
    np.random.seed(42)
    dates = pd.date_range('2017-01-01', '2024-01-01', freq='M')
    
    data_list = []
    minerals = ['Copper', 'Lithium', 'Graphite']
    
    for date in dates:
        for mineral in minerals:
            base_value = {'Copper': 5000, 'Lithium': 2000, 'Graphite': 1500}[mineral]
            trend = date.year - 2017
            seasonal = 500 * np.sin(2 * np.pi * date.month / 12)
            noise = np.random.normal(0, 200)
            
            import_value = base_value + (trend * 200) + seasonal + noise
            
            data_list.append({
                'date': date,
                'mineral': mineral,
                'year': date.year,
                'quarter': date.quarter,
                'month': date.month,
                'import_value': import_value
            })
    
    df = pd.DataFrame(data_list)
    
    print(f"\n✓ Loaded {len(df)} observations")
    print(f"✓ Minerals: {df['mineral'].nunique()}")
    print(f"✓ Time range: {df['date'].min()} to {df['date'].max()}")
    
    # Perform comprehensive analysis
    analyzer = EXIMANOVAAnalysis(df)
    
    analyzer.analyze_minerals_across_years()
    analyzer.analyze_minerals_comparison()
    analyzer.analyze_seasonal_patterns()
    analyzer.analyze_two_way_interaction()
    
    # Generate final report
    report = analyzer.generate_report()
    
    print("\n" + "="*70)
    print("ANOVA ANALYSIS COMPLETED SUCCESSFULLY")
    print("="*70)
    print("\nGenerated files:")
    print("  • anova_summary_report.csv")
    print("  • minerals_comparison_anova.png")
    print("  • seasonal_patterns_anova.png")
    print("  • mineral_year_interaction.png")
    print("  • [Mineral]_by_year_anova.png (for each mineral)")

In [ ]:
if __name__ == "__main__":
    main()